# Лекция 04: выделение ключевых особенностей / Lecture 04: Feature Detection

**Реальные данные: BSDS500. / Real data: BSDS500.**
Градиенты → края → углы → линии. / Gradients → edges → corners → lines.

[Open in Colab](https://colab.research.google.com/github/alexander-toschev/cv-course/blob/main/Lectures/04_FeatureDetection/Practice_RU_EN.ipynb)

Этот ноутбук объединяет темы авторских `FD_Edge_Detection`, `FD_PointsAndPatches_HARRIS`, `FD_Shi_Tomasi_CornerDetector`, `FD_Line_Detection_Hough`. Новое: единый датасет, воспроизводимость и количественная оценка. / This notebook combines the instructor's original detector topics with reproducible data and quantitative evaluation.

## Данные и условия / Data and terms
[BSDS500 official source](https://www2.eecs.berkeley.edu/Research/Projects/CS/vision/grouping/resources.html). Cite: Arbelaez, Maire, Fowlkes, Malik, *Contour Detection and Hierarchical Image Segmentation*, TPAMI 2011. Dataset origins: Martin et al., ICCV 2001. [Original BSDS terms](https://www2.eecs.berkeley.edu/Research/Projects/CS/vision/bsds/) describe non-commercial research and educational use. Images retain their original rights; this notebook downloads from the authors and does not relicense the dataset. / Учебное некоммерческое использование, исходные права сохраняются; архив не включён в репозиторий.

500 фото, официальные train/val/test = 200/100/200. Для занятия выбраны 3 train, 3 val, 3 test. Это небольшой учебный эксперимент, не результат на полном benchmark. / The fixed nine-image subset is a teaching experiment, not a full benchmark.

Не подбирайте параметры по test. Разметка описывает границы, а не классы объектов, углы или прямые. / Do not tune on test; annotations describe boundaries, not object classes, corners or lines.

## 1. Окружение / Environment
Python 3.10+; NumPy 1.26–2.x, OpenCV 4.10–5.x, SciPy 1.11–1.x, Matplotlib 3.8–3.x.
Если библиотек нет, выполните отдельной ячейкой и перезапустите ядро / If needed, run in a separate cell and restart the kernel:
```python
%pip install "numpy>=1.26,<3" "opencv-python-headless>=4.10,<6" "scipy>=1.11,<2" "matplotlib>=3.8,<4"
```
В ноутбуке нет установки при каждом запуске. / Installation is not repeated automatically.

In [ ]:
import os, platform
import scipy, matplotlib
import matplotlib.pyplot as plt
from pathlib import Path
import hashlib, json, tarfile, urllib.request
import cv2
import numpy as np
from scipy.io import loadmat
from scipy.spatial import cKDTree
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import maximum_bipartite_matching

DATA_URL = 'https://www2.eecs.berkeley.edu/Research/Projects/CS/vision/grouping/BSR/BSR_bsds500.tgz'
ARCHIVE_SHA256 = '97e49d31764f3912f0c4122707d53062ac9e783ba0f095e447a4d53c1a41af8e'
SPLITS = {'train':['100075','108073','118035'], 'val':['101085','101087','102061'], 'test':['100007','100039','100099']}

def prepare_data(cache):
    cache = Path(cache); cache.mkdir(parents=True, exist_ok=True)
    archive = cache / 'BSR_bsds500.tgz'
    if not archive.exists():
        temp = archive.with_suffix('.download')
        try:
            with urllib.request.urlopen(DATA_URL, timeout=120) as src, temp.open('wb') as dst:
                import shutil
                shutil.copyfileobj(src, dst)
        except Exception as exc:
            raise RuntimeError('BSDS500 unavailable. Retry official download; do not replace with synthetic data.') from exc
        if hashlib.sha256(temp.read_bytes()).hexdigest() != ARCHIVE_SHA256:
            raise ValueError('Unexpected archive checksum; keep the file for inspection and do not extract.')
        temp.replace(archive)
    if hashlib.sha256(archive.read_bytes()).hexdigest() != ARCHIVE_SHA256:
        raise ValueError('Archive checksum mismatch')
    root = cache / 'BSR/BSDS500/data'
    with tarfile.open(archive) as tar:
        for split, ids in SPLITS.items():
            for iid in ids:
                for kind, ext in [('images','jpg'), ('groundTruth','mat')]:
                    name = f'BSR/BSDS500/data/{kind}/{split}/{iid}.{ext}'
                    dest = cache / name
                    member = tar.getmember(name)
                    if not member.isfile(): raise ValueError('Expected regular file')
                    source = tar.extractfile(member).read()
                    if dest.exists():
                        if hashlib.sha256(dest.read_bytes()).digest() != hashlib.sha256(source).digest():
                            raise ValueError(f'Cached dataset file changed: {dest}')
                    else:
                        dest.parent.mkdir(parents=True, exist_ok=True)
                        dest.write_bytes(source)
    return root

def load_example(root, split, iid):
    if iid not in SPLITS[split]: raise ValueError('Use the frozen teaching subset')
    bgr = cv2.imread(str(Path(root)/'images'/split/f'{iid}.jpg'))
    if bgr is None: raise ValueError('Unreadable image')
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    gray = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)
    records = loadmat(Path(root)/'groundTruth'/split/f'{iid}.mat')['groundTruth'].ravel()
    annotations = [np.asarray(r['Boundaries'][0,0], dtype=bool) for r in records]
    assert gray.dtype == np.uint8 and gray.ndim == 2
    assert annotations and all(a.shape == gray.shape and a.any() for a in annotations)
    return rgb, gray, annotations

def gradients(gray, operator='sobel'):
    if gray.ndim != 2 or gray.dtype != np.uint8: raise ValueError('Expected grayscale uint8')
    src = gray.astype(np.float32) / 255
    if operator == 'sobel':
        gx = cv2.Sobel(src, cv2.CV_32F, 1, 0, ksize=3, scale=1/8, borderType=cv2.BORDER_REFLECT_101)
        gy = cv2.Sobel(src, cv2.CV_32F, 0, 1, ksize=3, scale=1/8, borderType=cv2.BORDER_REFLECT_101)
    elif operator == 'scharr':
        gx = cv2.Scharr(src, cv2.CV_32F, 1, 0, scale=1/32, borderType=cv2.BORDER_REFLECT_101)
        gy = cv2.Scharr(src, cv2.CV_32F, 0, 1, scale=1/32, borderType=cv2.BORDER_REFLECT_101)
    else: raise ValueError('Unknown derivative operator')
    return gx, gy, np.hypot(gx,gy), np.arctan2(gy,gx)

def canny(gray, low=50, high=100, sigma=1.0):
    if gray.ndim != 2 or gray.dtype != np.uint8: raise ValueError('Expected grayscale uint8')
    if not 0 <= low < high or sigma <= 0: raise ValueError('Invalid Canny parameters')
    smooth = cv2.GaussianBlur(gray, (0,0), sigmaX=float(sigma), borderType=cv2.BORDER_REFLECT_101)
    return cv2.Canny(smooth, float(low), float(high), apertureSize=3, L2gradient=True).astype(bool)

def boundary_prf(pred, truth, tolerance=2.0):
    """Teaching metric: maximum-cardinality one-to-one pixel matching.
    Euclidean tolerance in original-image pixels. Not the official BSDS metric.
    Empty/empty gives (1,1,1); one empty gives (0,0,0).
    """
    pred, truth = np.asarray(pred,dtype=bool), np.asarray(truth,dtype=bool)
    if pred.shape != truth.shape or pred.ndim != 2 or tolerance < 0: raise ValueError('Invalid boundary maps')
    a, b = np.argwhere(pred), np.argwhere(truth)
    if not len(a) and not len(b): return np.ones(3)
    if not len(a) or not len(b): return np.zeros(3)
    neighbors = cKDTree(b).query_ball_point(a, tolerance)
    sizes = np.array([len(x) for x in neighbors])
    rr = np.repeat(np.arange(len(a)), sizes)
    cc = np.array([j for row in neighbors for j in row],dtype=int)
    graph = csr_matrix((np.ones(len(cc),dtype=np.int8),(rr,cc)),shape=(len(a),len(b)))
    count = np.count_nonzero(maximum_bipartite_matching(graph,perm_type='column') >= 0)
    precision, recall = count/len(a), count/len(b)
    f1 = 2*precision*recall/(precision+recall) if count else 0
    return np.array([precision,recall,f1])

def evaluate(root, split, params):
    rows=[]
    for iid in SPLITS[split]:
        _, gray, anns = load_example(root,split,iid)
        edges = canny(gray,**params)
        prf = np.mean([boundary_prf(edges,a) for a in anns],axis=0)
        rows.append({'image':iid,'annotators':len(anns),'precision':float(prf[0]),'recall':float(prf[1]),'f1':float(prf[2])})
    return rows

def corners(gray, harris=False):
    # Unified point selection: same qualityLevel and minDistance for both scores.
    points = cv2.goodFeaturesToTrack(gray, maxCorners=80, qualityLevel=.02, minDistance=8,
                                    blockSize=3, useHarrisDetector=harris, k=.04)
    return np.empty((0,2),dtype=np.float32) if points is None else points.reshape(-1,2)

def line_segments(gray):
    edges = canny(gray)
    lines = cv2.HoughLinesP(edges.astype(np.uint8)*255,1,np.pi/180,threshold=35,minLineLength=40,maxLineGap=8)
    return np.empty((0,4),dtype=int) if lines is None else lines.reshape(-1,4)

cv2.setNumThreads(1)
cv2.setRNGSeed(0)
print({'python': platform.python_version(), 'numpy': np.__version__, 'opencv': cv2.__version__,
       'scipy': scipy.__version__, 'matplotlib': matplotlib.__version__})


## 2. Загрузка / Download
Архив около 67 MiB, SHA-256 фиксирован. При ошибке загрузки исправляем доступ к официальному источнику; синтетической замены нет. Переменная `CV_BSDS_CACHE` задаёт папку кэша. / The official archive is checksum-verified; no synthetic fallback. `CV_BSDS_CACHE` overrides the local cache path.

In [ ]:
ROOT = prepare_data(Path(os.environ.get('CV_BSDS_CACHE', 'data/bsds500')))
print(json.dumps(SPLITS, indent=2))
rgb, gray, annotations = load_example(ROOT, 'train', '118035')
print('shape/dtype:', gray.shape, gray.dtype, 'annotators:', len(annotations))
fig, ax = plt.subplots(1, 2, figsize=(12, 5))
ax[0].imshow(rgb); ax[0].set_title('BSDS500 train/118035')
ax[1].imshow(np.mean(annotations, axis=0), cmap='gray', vmin=0, vmax=1)
ax[1].set_title('Mean human boundary agreement')
for a in ax: a.axis('off')
plt.show()

## 3. Градиенты / Gradients
`gx, gy` сохраняют знак. `hypot(gx, gy)` — модуль, `atan2(gy, gx)` — направление. `Sobel(dx=1, dy=1)` является смешанной производной, а не модулем. / The mixed derivative is not gradient magnitude.
Sobel 3×3 масштабируется на 1/8, Scharr на 1/32 для сопоставимого отклика на линейный перепад. Общая шкала отображения обязательна. / Scales compensate different kernel gains; use a common display range.

In [ ]:
gx, gy, mag, angle = gradients(gray, 'sobel')
_, _, scharr_mag, _ = gradients(gray, 'scharr')
fig, ax = plt.subplots(1, 3, figsize=(15, 5))
limit = max(float(mag.max()), float(scharr_mag.max()))
for a, im, title in zip(ax, [gx, mag, scharr_mag], ['Signed Gx', 'Sobel magnitude', 'Scharr magnitude']):
    a.imshow(im, cmap='coolwarm' if title=='Signed Gx' else 'gray',
             vmin=-limit if title=='Signed Gx' else 0, vmax=limit)
    a.set_title(title); a.axis('off')
plt.show()
assert np.allclose(mag, np.hypot(gx, gy)) and np.isfinite(angle).all()

## 4. Canny на реальном изображении / Canny on a real image
Явное GaussianBlur перед `Canny`, `L2gradient=True`. Меняем только пороги при фиксированном σ. / Apply Gaussian smoothing explicitly and vary thresholds with sigma fixed.
На тигре границы полос не равны границам самого животного. / Tiger stripes demonstrate why image edges are not object boundaries.

In [ ]:
tiger_rgb, tiger, tiger_gt = load_example(ROOT, 'train', '108073')
fig, ax = plt.subplots(1, 4, figsize=(16, 4))
ax[0].imshow(tiger_rgb); ax[0].set_title('train/108073')
for a, pair in zip(ax[1:], [(20,40),(50,100),(100,200)]):
    a.imshow(canny(tiger,*pair),cmap='gray'); a.set_title(f'Canny {pair[0]}/{pair[1]}')
for a in ax: a.axis('off')
plt.show()

## 5. Harris и Shi–Tomasi / Harris and Shi–Tomasi
Harris: `det(M) − k·trace(M)²`; Shi–Tomasi: `min(λ₁, λ₂)`. Сравниваем точки с одинаковыми правилами отбора: максимум 80, qualityLevel 0.02, minDistance 8. / Both scores use the same point-selection settings. Relative quality thresholds still do not make score scales identical.
Не считайте красные пиксели на карте Harris числом независимых углов. / A thresholded response map can contain many pixels per corner.

In [ ]:
fig, ax = plt.subplots(1,2,figsize=(12,5))
for a, use_harris, label in zip(ax, [True,False], ['Harris','Shi–Tomasi']):
    points = corners(gray, harris=use_harris)
    a.imshow(rgb)
    if len(points): a.scatter(points[:,0],points[:,1],s=22,facecolors='none',edgecolors='red')
    a.set_title(f'{label}: {len(points)} points'); a.axis('off')
plt.show()
response = cv2.cornerHarris(gray.astype(np.float32)/255,3,3,.04)
assert response.shape == gray.shape and np.isfinite(response).all()

## 6. Hough: линии и отрезки / Hough: lines and segments
`ρ = x cos θ + y sin θ`, θ — угол нормали. HoughLinesP возвращает `(x1,y1,x2,y2)`. / Theta is the normal angle; the probabilistic implementation returns endpoints.
Геометрические линии не являются разметкой объектов. Крыша может иметь много отрезков, а их объединение требует отдельного правила. / Several detected segments may represent one roof edge.

In [ ]:
segments = line_segments(gray)
fig, ax = plt.subplots(figsize=(10,6)); ax.imshow(rgb)
for x1,y1,x2,y2 in segments: ax.plot([x1,x2],[y1,y2],color='red',linewidth=1.5)
ax.set_title(f'train/118035: {len(segments)} Hough segments'); ax.axis('off'); plt.show()

## 7. Учебная оценка границ / Teaching boundary metric
Мы используем максимальное взаимно-однозначное сопоставление пикселей в радиусе **2 пикселя** исходного изображения. Один предсказанный пиксель не покрывает несколько эталонных. / Maximum-cardinality one-to-one matching within 2 original-image pixels prevents duplicate credit.
Для каждого аннотатора считаются P, R, F1; затем отдельно усредняются по аннотаторам и изображениям. Средний F1 не обязан совпадать с F1 из средних P и R. Пустая пара даёт 1; одна пустая карта даёт 0. / Average per-annotator metrics and then per-image metrics. Empty/empty=1; one empty=0.
**Это не официальный BSDS benchmark:** иной алгоритм сопоставления и агрегации, фиксированный допуск, всего 9 изображений, нет ODS/OIS/AP. Для научного сравнения используйте официальный evaluator. / Do not compare these teaching scores with published BSDS ODS/OIS/AP.

In [ ]:
truth = annotations[0]
assert np.allclose(boundary_prf(truth, truth), [1,1,1])
empty = truth & False  # Empty detector output for a real image, not a synthetic dataset.
assert np.allclose(boundary_prf(empty, truth), [0,0,0])
half = truth.copy(); half[:, ::2] = False
p, r, f = boundary_prf(half,truth,tolerance=0)
assert np.isclose(p,1) and np.isclose(r,half.sum()/truth.sum()) and 0<=f<=1
pred = canny(gray)
print('Per-annotator P/R/F1:', [boundary_prf(pred,a).round(3).tolist() for a in annotations])

## 8. Настройка только на validation / Validation-only tuning
Фиксированная небольшая сетка: 3 пары порогов × 2 масштаба. Критерий — средний учебный F1. / Fixed six-configuration grid, selected by mean teaching F1. No claims about statistical significance from three validation images.

In [ ]:
candidates = [dict(low=lo,high=hi,sigma=s) for s in [1.0,1.6] for lo,hi in [(20,40),(50,100),(100,200)]]
validation = []
for params in candidates:
    rows = evaluate(ROOT,'val',params)
    validation.append({'params':params,'mean_f1':float(np.mean([r['f1'] for r in rows]))})
best = max(validation,key=lambda row:row['mean_f1'])
BEST_PARAMS = dict(best['params'])
print(json.dumps(validation,indent=2))
print('Frozen parameters:', BEST_PARAMS)
frozen = {'assignment':'Lecture04_practice','dataset':'BSDS500','archive_sha256':ARCHIVE_SHA256,
          'splits':SPLITS,'metric':'teaching_one_to_one_2px_macro','params':BEST_PARAMS}
Path('lecture04_frozen_parameters.json').write_text(json.dumps(frozen,indent=2))

## 9. Итоговая оценка / Final evaluation
Следующую ячейку выполняем после фиксации параметров, не возвращаясь к подбору по test. Если меняете алгоритм после просмотра test, называйте это исследованием, а не независимой оценкой. / Evaluate after freezing parameters; changing the algorithm after viewing test invalidates the held-out claim.

In [ ]:
test_rows = evaluate(ROOT,'test',BEST_PARAMS)
result = {**frozen,'validation':validation,'test':test_rows,
          'test_mean_f1':float(np.mean([r['f1'] for r in test_rows])),
          'versions':{'opencv':cv2.__version__,'numpy':np.__version__,'scipy':scipy.__version__}}
Path('lecture04_results.json').write_text(json.dumps(result,indent=2))
print(json.dumps({'test':test_rows,'test_mean_f1':result['test_mean_f1']},indent=2))

## 10. Практическое задание / Practice assignment
Это упражнение на занятии. Отдельное [ДЗ №4 RU/EN](Homework_RU_EN.ipynb) находится рядом с практикой. / This is a classroom exercise. The separate [Homework 4 RU/EN](Homework_RU_EN.ipynb) is in the same folder.

1. На train/108073 объясните, почему при низких порогах появляются полосы тигра и фон. / Explain tiger stripes and background edges at low thresholds.
2. На train/118035 сравните Sobel/Scharr на общей шкале, Harris/Shi–Tomasi при одинаковых параметрах отбора. / Compare derivatives and point detectors fairly.
3. Для Hough измените один параметр на train, покажите лишний и пропущенный отрезок. / Change one Hough parameter on train and discuss one false and one missed segment.
4. Приложите `lecture04_results.json`, версии библиотек, три иллюстрации и 5–8 предложений о причинах ошибок. / Include the JSON result, versions, three figures and a short error analysis.
5. Выберите реальную фотографию train для следующего занятия и объясните, какие точки стоит сопоставлять. / Select a real train photograph and explain which points could be matched.

Автоматические проверки подтверждают корректность формы данных, метрики и воспроизводимость; качество объяснений оценивается преподавателем. / Assertions verify contracts and metric calculations; explanations require instructor review.

### Источники / References
- [Sobel / Scharr](https://docs.opencv.org/4.x/d2/d2c/tutorial_sobel_derivatives.html)
- [Canny](https://docs.opencv.org/4.x/da/d22/tutorial_py_canny.html)
- [Harris](https://docs.opencv.org/4.x/dc/d0d/tutorial_py_features_harris.html)
- [Shi–Tomasi](https://docs.opencv.org/4.x/d4/d8c/tutorial_py_shi_tomasi.html)
- [Hough lines](https://docs.opencv.org/4.x/d9/db0/tutorial_hough_lines.html)
